# Object Detection con YOLOv8

En este notebook exploraremos la **deteccion de objetos** usando YOLOv8 (You Only Look Once), uno de los modelos
mas populares y eficientes para detectar y localizar objetos en imagenes y video en tiempo real.

**Objetivos:**
- Entender los conceptos fundamentales de deteccion de objetos
- Usar un modelo YOLOv8 preentrenado para inferencia
- Entrenar un modelo personalizado con datos propios
- Evaluar el rendimiento del modelo
- Exportar el modelo para produccion

## Conceptos fundamentales de deteccion de objetos

### Bounding Boxes
Un **bounding box** es un rectangulo que enmarca un objeto detectado. Se define por sus coordenadas:
- Formato YOLO: `(x_center, y_center, width, height)` normalizado entre 0 y 1
- Formato Pascal VOC: `(x_min, y_min, x_max, y_max)` en pixeles
- Formato COCO: `(x_min, y_min, width, height)` en pixeles

### IoU (Intersection over Union)
El **IoU** mide el solapamiento entre la prediccion y el ground truth:

$$IoU = \frac{Area\_Interseccion}{Area\_Union}$$

- IoU = 1.0: prediccion perfecta
- IoU > 0.5: generalmente considerada una buena deteccion
- IoU = 0.0: sin solapamiento

### NMS (Non-Maximum Suppression)
**NMS** es una tecnica de post-procesamiento que elimina detecciones duplicadas:
1. Ordenar detecciones por confianza (de mayor a menor)
2. Tomar la deteccion con mayor confianza
3. Eliminar todas las detecciones restantes que tengan IoU > umbral con la seleccionada
4. Repetir hasta procesar todas las detecciones

### mAP (Mean Average Precision)
**mAP** es la metrica estandar para evaluar modelos de deteccion:
- **Precision**: De todas las detecciones, cuantas son correctas
- **Recall**: De todos los objetos reales, cuantos fueron detectados
- **AP**: Area bajo la curva Precision-Recall para una clase
- **mAP50**: Media de AP calculada con IoU >= 0.5
- **mAP50-95**: Media de AP calculada con IoU desde 0.5 hasta 0.95 (paso 0.05). Es la metrica mas exigente.

In [ ]:
# Install ultralytics (YOLOv8 framework)
# !pip install ultralytics

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

# Load a pretrained YOLOv8 nano model (smallest and fastest)
# Available sizes: yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
model = YOLO("yolov8n.pt")

# Model summary: parameters, layers, and architecture
print(f"Model: {model.model.__class__.__name__}")
print(f"Task: {model.task}")
print(f"Number of classes: {len(model.names)}")
print(f"\nFirst 20 classes:")
for i in range(20):
    print(f"  {i}: {model.names[i]}")

In [ ]:
import urllib.request
import os

# Download a sample image for inference
sample_url = "https://ultralytics.com/images/bus.jpg"
sample_path = "sample_bus.jpg"

if not os.path.exists(sample_path):
    urllib.request.urlretrieve(sample_url, sample_path)
    print(f"Downloaded sample image to {sample_path}")

# Run inference
results = model(sample_path, conf=0.25)  # confidence threshold of 0.25

# Access the first result (one image = one result)
result = results[0]

# Print detection summary
print(f"Detected {len(result.boxes)} objects:\n")
for box in result.boxes:
    cls_id = int(box.cls[0])
    cls_name = model.names[cls_id]
    confidence = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    print(f"  {cls_name}: {confidence:.2f} at [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")

# Visualize results with bounding boxes
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Original image
img = Image.open(sample_path)
axes[0].imshow(img)
axes[0].set_title("Imagen original", fontsize=14)
axes[0].axis("off")

# Image with detections
annotated = result.plot()  # Returns BGR numpy array with annotations
axes[1].imshow(annotated[..., ::-1])  # Convert BGR to RGB for matplotlib
axes[1].set_title("Detecciones YOLOv8", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Manual bounding box visualization for educational purposes
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.imshow(img)

colors = plt.cm.Set3(np.linspace(0, 1, 12))
for i, box in enumerate(result.boxes):
    cls_id = int(box.cls[0])
    cls_name = model.names[cls_id]
    confidence = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    
    color = colors[cls_id % len(colors)]
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor=color, facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(
        x1, y1 - 5, f"{cls_name} {confidence:.2f}",
        fontsize=10, color="white",
        bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8)
    )

ax.set_title("Bounding Boxes manuales", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## Entrenar con datos custom

Para entrenar YOLOv8 con tus propios datos necesitas:

1. **Imagenes** organizadas en carpetas `train/`, `val/` y opcionalmente `test/`
2. **Etiquetas** en formato YOLO (un archivo `.txt` por imagen)
3. Un archivo **YAML** que describe el dataset

### Formato de etiquetas YOLO
Cada linea del archivo `.txt` contiene:
```
<class_id> <x_center> <y_center> <width> <height>
```
Donde todos los valores estan **normalizados** entre 0 y 1.

### Estructura de carpetas
```
dataset/
  images/
    train/
      img001.jpg
      img002.jpg
    val/
      img003.jpg
  labels/
    train/
      img001.txt
      img002.txt
    val/
      img003.txt
  data.yaml
```

In [ ]:
# Example: Dataset YAML structure for custom training
dataset_yaml_example = """
# data.yaml - Dataset configuration file
# All paths are relative to this YAML file or absolute

path: ./dataset          # Root directory of the dataset
train: images/train      # Training images (relative to 'path')
val: images/val          # Validation images (relative to 'path')
test: images/test        # Test images (optional)

# Number of classes
nc: 3

# Class names (must match class_id in label files)
names:
  0: helmet
  1: vest
  2: person
"""
print("Ejemplo de archivo data.yaml:")
print(dataset_yaml_example)

# Example: What a label file looks like
label_example = """
# img001.txt - Each line is one object
# class_id  x_center  y_center  width  height  (all normalized 0-1)
0 0.4521 0.3240 0.1200 0.0980
2 0.4500 0.5500 0.3000 0.7000
1 0.4480 0.4200 0.2500 0.2000
"""
print("Ejemplo de archivo de etiquetas (img001.txt):")
print(label_example)

# Visualize what these coordinates mean
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.set_xlim(0, 1)
ax.set_ylim(1, 0)  # Invert y-axis (image coordinates)
ax.set_aspect("equal")
ax.set_title("Formato de coordenadas YOLO (normalizado)", fontsize=12)

# Draw example bounding boxes
examples = [
    (0, 0.4521, 0.3240, 0.1200, 0.0980, "helmet", "red"),
    (2, 0.4500, 0.5500, 0.3000, 0.7000, "person", "blue"),
    (1, 0.4480, 0.4200, 0.2500, 0.2000, "vest", "green"),
]

for cls_id, xc, yc, w, h, name, color in examples:
    x1 = xc - w / 2
    y1 = yc - h / 2
    rect = patches.Rectangle(
        (x1, y1), w, h,
        linewidth=2, edgecolor=color, facecolor=color, alpha=0.2
    )
    ax.add_patch(rect)
    ax.plot(xc, yc, "o", color=color, markersize=6)  # Center point
    ax.text(x1, y1 - 0.01, f"{name} (cls={cls_id})", fontsize=9, color=color)

ax.set_xlabel("x (normalizado)")
ax.set_ylabel("y (normalizado)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train YOLOv8 on COCO8 - a small 8-image subset included with ultralytics
# This is perfect for testing the training pipeline quickly

from ultralytics import YOLO

# Load a fresh YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Train on COCO8 (auto-downloaded, 4 train + 4 val images)
# In production, you'd use epochs=100+ and a larger dataset
results = model.train(
    data="coco8.yaml",    # Built-in dataset config
    epochs=10,            # Few epochs for demonstration
    imgsz=640,            # Input image size
    batch=4,              # Batch size (small for demo)
    device="cpu",         # Use 'cuda' or '0' for GPU
    project="runs/detect",
    name="coco8_demo",
    verbose=True,
)

print("\nEntrenamiento completado!")
print(f"Resultados guardados en: {results.save_dir}")

In [ ]:
# Evaluate the trained model
from ultralytics import YOLO
import os

# Load the trained model (best weights)
best_model_path = "runs/detect/coco8_demo/weights/best.pt"
if os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)
else:
    # Fallback to pretrained if training hasn't been run
    print("Modelo entrenado no encontrado, usando pretrained para demo...")
    trained_model = YOLO("yolov8n.pt")

# Run validation to get metrics
metrics = trained_model.val(
    data="coco8.yaml",
    imgsz=640,
    device="cpu",
)

# Print key metrics
print("=" * 50)
print("METRICAS DE EVALUACION")
print("=" * 50)
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

# Per-class metrics
print(f"\nMetricas por clase:")
print(f"{'Clase':<15} {'mAP50':>8} {'mAP50-95':>10}")
print("-" * 35)
for i, ap50 in enumerate(metrics.box.ap50):
    if i < len(metrics.box.ap):
        cls_name = trained_model.names.get(i, f"class_{i}")
        print(f"{cls_name:<15} {ap50:>8.4f} {metrics.box.ap[i]:>10.4f}")

# Visualize predictions vs ground truth on validation images
val_results = trained_model.predict(
    source="coco8.yaml",
    imgsz=640,
    conf=0.25,
    save=True,
    project="runs/detect",
    name="coco8_predictions",
    device="cpu",
)

# Show some prediction examples if available
pred_dir = "runs/detect/coco8_predictions"
if os.path.exists(pred_dir):
    pred_images = [f for f in os.listdir(pred_dir) if f.endswith((".jpg", ".png"))]
    n_show = min(4, len(pred_images))
    if n_show > 0:
        fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
        if n_show == 1:
            axes = [axes]
        for ax, img_name in zip(axes, pred_images[:n_show]):
            img = Image.open(os.path.join(pred_dir, img_name))
            ax.imshow(img)
            ax.set_title(img_name, fontsize=10)
            ax.axis("off")
        plt.suptitle("Predicciones del modelo entrenado", fontsize=14)
        plt.tight_layout()
        plt.show()

## Exportar modelo para produccion

Una vez entrenado el modelo, es comun exportarlo a formatos optimizados para inferencia en produccion:

| Formato | Uso principal | Ventaja |
|---------|--------------|----------|
| **ONNX** | Multiplataforma | Compatible con muchos runtimes |
| **TensorRT** | GPU NVIDIA | Maxima velocidad en GPU |
| **CoreML** | iOS/macOS | Nativo en dispositivos Apple |
| **TFLite** | Mobile/Edge | Optimizado para dispositivos moviles |
| **OpenVINO** | Intel CPUs | Optimizado para hardware Intel |

ONNX es el formato mas versatil y recomendado como punto de partida.

In [ ]:
# Export model to ONNX format
# !pip install onnx onnxruntime

import time
import numpy as np

# Load the model to export
model = YOLO("yolov8n.pt")

# Export to ONNX
onnx_path = model.export(
    format="onnx",
    imgsz=640,
    simplify=True,    # Simplify the ONNX graph
    dynamic=False,    # Fixed input size for maximum speed
)
print(f"\nModelo exportado a: {onnx_path}")

# Compare file sizes
import os
pt_size = os.path.getsize("yolov8n.pt") / (1024 * 1024)
onnx_size = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"\nTamano PyTorch: {pt_size:.1f} MB")
print(f"Tamano ONNX:    {onnx_size:.1f} MB")

# Run inference with ONNX Runtime
try:
    import onnxruntime as ort

    # Create ONNX Runtime session
    session = ort.InferenceSession(onnx_path)
    input_name = session.get_inputs()[0].name
    input_shape = session.get_inputs()[0].shape
    print(f"\nONNX input: {input_name}, shape: {input_shape}")

    # Prepare dummy input for benchmarking
    dummy_input = np.random.randn(1, 3, 640, 640).astype(np.float32)

    # Benchmark: PyTorch vs ONNX Runtime
    n_runs = 20

    # ONNX Runtime speed
    # Warmup
    for _ in range(3):
        session.run(None, {input_name: dummy_input})

    start = time.time()
    for _ in range(n_runs):
        session.run(None, {input_name: dummy_input})
    onnx_time = (time.time() - start) / n_runs * 1000

    # PyTorch speed (using YOLO predict with a dummy)
    dummy_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    # Warmup
    for _ in range(3):
        model.predict(dummy_img, verbose=False)

    start = time.time()
    for _ in range(n_runs):
        model.predict(dummy_img, verbose=False)
    pt_time = (time.time() - start) / n_runs * 1000

    print(f"\n{'='*40}")
    print(f"BENCHMARK (promedio de {n_runs} ejecuciones)")
    print(f"{'='*40}")
    print(f"PyTorch:      {pt_time:.1f} ms/imagen")
    print(f"ONNX Runtime: {onnx_time:.1f} ms/imagen")
    print(f"Speedup:      {pt_time/onnx_time:.2f}x")

except ImportError:
    print("\nInstala onnxruntime para comparar velocidades:")
    print("pip install onnxruntime")

## Resumen y aplicaciones de negocio

YOLOv8 es una herramienta poderosa y accesible para deteccion de objetos. Su velocidad y precision
lo hacen ideal para muchas aplicaciones empresariales.

### Aplicaciones de negocio

| Industria | Caso de uso | Descripcion |
|-----------|------------|-------------|
| **Retail** | Conteo de personas | Medir trafico en tiendas, optimizar layouts |
| **Manufactura** | Control de calidad | Detectar defectos en lineas de produccion |
| **Seguridad** | Vigilancia inteligente | Detectar intrusos, objetos abandonados |
| **Logistica** | Gestion de inventario | Contar productos en estanterias automaticamente |
| **Agricultura** | Deteccion de plagas | Identificar enfermedades en cultivos con drones |
| **Salud** | Analisis de imagenes medicas | Detectar anomalias en radiografias |
| **Construccion** | Seguridad laboral | Verificar uso de EPP (casco, chaleco) |
| **Transporte** | Conduccion autonoma | Detectar vehiculos, peatones, senales |
| **Medio ambiente** | Monitoreo de fauna | Contar y clasificar especies animales |

### Checklist de proyecto de deteccion de objetos

- [ ] Definir clases a detectar
- [ ] Recopilar y anotar imagenes (minimo 100-300 por clase)
- [ ] Elegir tamano de modelo segun restricciones (nano, small, medium, large)
- [ ] Entrenar con data augmentation apropiada
- [ ] Evaluar con mAP50-95 y revisar falsos positivos/negativos
- [ ] Exportar a formato de produccion (ONNX, TensorRT, etc.)
- [ ] Optimizar velocidad para el hardware destino
- [ ] Implementar pipeline de monitoreo de modelo en produccion